In [ ]:
import random, uuid

customers_int = 10
ts_step_int = 1

class OrderGenerator:
    def __init__(self):
        self.customer_id_int = 0
        #
        self.ts_int = 0
        self.ts_step_int = ts_step_int

    def generate(self):
        order_id_str = str(uuid.uuid4())
        message_dict = {
            "key": order_id_str,
            "value": {"order_id": order_id_str,
                      "customer_id": random.randint(0, customers_int - 1),
                      "price": random.randint(1, 10000) / 100,
                      "ts": self.ts_int},
        }
        #
        self.ts_int += self.ts_step_int
        #
        return message_dict

#

order_source_str = "orders"
sink_str = "sink"

#

def push(built_tn, customer_id, price, ts, w=1):
    r = {"value": {"customer_id": customer_id, "price": price, "ts": ts}}
    built_tn.push("orders", [(r, w)])
    #
    sink_str_r_w_tuple_list_dict = built_tn.latest()
    #
    if sink_str_r_w_tuple_list_dict == {}:
        return []
    else:
        return sink_str_r_w_tuple_list_dict[sink_str]

def assert_output(actual_r_w_tuple_list, expected_r_w_tuple_list):
    def sort_key(r_w_tuple):
        r, w = r_w_tuple
        return (r.get("window_end", 0), r.get("customer_id", 0), w)
    #
    actual_sorted_r_w_tuple_list = sorted(actual_r_w_tuple_list, key=sort_key)
    expected_sorted_r_w_tuple_list = sorted(expected_r_w_tuple_list, key=sort_key)
    #
    if actual_sorted_r_w_tuple_list != expected_sorted_r_w_tuple_list:
        raise ValueError(f"\nAssertion Failed!\nExpected: {expected_sorted_r_w_tuple_list}\nGot:      {actual_sorted_r_w_tuple_list}")



In [ ]:
import sys
sys.path.insert(1, "..")

import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)

from kafi.streams.topologynode import TopologyNode as Tn

#

size_int = ts_step_int * 100
allowed_lateness_int = size_int * 3
#
order_source_tn = Tn.source(order_source_str)
order_source_tn.to_zSet(Tn._from_records)
order_tn = (
    order_source_tn
    .map(lambda r: {"customer_id": r["value"]["customer_id"],
                    "price": r["value"]["price"],
                    "ts": r["value"]["ts"]})
    .expire_tumbling(lambda r: r["ts"], size_int, allowed_lateness_int)
    .distinct()
)
#
window_tn = order_tn.window_tumbling(size_int,
                                     lambda r: r["ts"],
                                     lambda r: r["customer_id"],
                                     lambda agg_r, r: {"orders": agg_r["orders"] + 1,
                                                       "total_price": agg_r["total_price"] + r["price"]},
                                     {"orders": 0, "total_price": 0},
                                     lambda by, agg_r: {"customer_id": by,
                                                        "orders": agg_r["orders"],
                                                        "total_price": agg_r["total_price"]},
                                     lambda r_int_tuple: {**r_int_tuple[0], "window_end": r_int_tuple[1]},
                                     trigger_positive_only=False)
#
built_tn = Tn.build(window_tn.sink(sink_str))
built_tn.from_zSet(Tn._to_records)

In [ ]:
built_tn.reset()

print("=== Step 1: Two orders from customer 1 (price=100, ts=10) and (price=200, ts=50) arrive ===")
push(built_tn, customer_id=1, price=100, ts=10, w=1)
push(built_tn, customer_id=1, price=200, ts=50, w=1)
print("-> OK")

print("\n=== Step 2: An order from customer 2 (price=50, ts=105) arrives ===")
r_w_tuple_list = push(built_tn, customer_id=2, price=50, ts=105, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 1, "orders": 2, "total_price": 300, "window_end": 100}, 1)
])
print("-> OK: Window [0, 100) triggered (orders=2, total=300).")

print("\n=== Step 3: The order from customer 1 at 50 is retracted ===")
r_w_tuple_list = push(built_tn, customer_id=1, price=200, ts=50, w=-1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 1, "orders": 2, "total_price": 300, "window_end": 100}, -1),
    ({"customer_id": 1, "orders": 1, "total_price": 100, "window_end": 100}, 1)
])
print("-> OK: Correction for window [0, 100) triggered: (customer=1, orders=1, total=100).")

print("\n=== Step 4: The order from customer 1 at 10 is also retracted ===")
r_w_tuple_list = push(built_tn, customer_id=1, price=100, ts=10, w=-1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 1, "orders": 1, "total_price": 100, "window_end": 100}, -1)
])
print("-> OK: Retraction for window [0, 100) triggered.")

print("\n=== Step 5: An order from customer 3 (price=400, ts=100) arrives ===")
r_w_tuple_list = push(built_tn, customer_id=3, price=400, ts=100, w=1)
assert_output(r_w_tuple_list, [])
print("-> OK: Order put into window [100, 200), still correctly held back/not triggered.")

print("\n=== Step 6: Another order from customer 3 at 150 (price=200, ts=150) arrives ===")
r_w_tuple_list = push(built_tn, customer_id=3, price=200, ts=150, w=1)
assert_output(r_w_tuple_list, [])
print("-> OK. Order put into window [100, 200), still correctly held back/not triggered.")

print("\n=== Step 7: Another order from customer 1 (price=50, ts=210) arrives ===")
r_w_tuple_list = push(built_tn, customer_id=1, price=50, ts=210, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 2, "orders": 1, "total_price": 50, "window_end": 200}, 1),
    ({"customer_id": 3, "orders": 2, "total_price": 600, "window_end": 200}, 1)
])
print("-> OK: Window [100, 200) triggered (customer=2, orders=1, total=50), (customer=3, orders=2, total=600).")

print("\n=== Step 8: An order from customer 2 (price=60, ts=310) arrives ===")
r_w_tuple_list = push(built_tn, customer_id=2, price=60, ts=310, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 1, "orders": 1, "total_price": 50, "window_end": 300}, 1)
])
print("-> OK: Window [200, 300) triggered (customer=1, orders=1, total=50).")

print("\n=== Step 9: Order from customer 2 (price=40, ts=120) arrives late (but not too late) ===")
r_w_tuple_list = push(built_tn, customer_id=2, price=40, ts=120, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 2, "orders": 1, "total_price": 50, "window_end": 200}, -1),
    ({"customer_id": 2, "orders": 2, "total_price": 90, "window_end": 200}, 1)
])
print("-> OK: Correction for window [100, 200) triggered: (customer=2, orders=2, total=90).")

print("\n=== Step 10: Order from customer 1 (price=70, ts=510) arrives ===")
r_w_tuple_list = push(built_tn, customer_id=1, price=70, ts=510, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 2, "orders": 2, "total_price": 90, "window_end": 200}, -1),
    ({"customer_id": 3, "orders": 2, "total_price": 600, "window_end": 200}, -1),
    ({"customer_id": 2, "orders": 1, "total_price": 60, "window_end": 400}, 1)
])
print("-> OK: Retraction for window [100, 200) triggered. Window (300, 400) triggered: (customer=2, orders=1, total=60).")

print("\n=== Step 11: Order from customer 1 (price=70, ts=130) arrives too late ===")
r_w_tuple_list = push(built_tn, customer_id=1, price=70, ts=130, w=1)
assert_output(r_w_tuple_list, [])
print("-> OK: Order arrived too late - no action.")

print("\n🎉 Done.")

In [ ]:
import sys
sys.path.insert(1, "..")

import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)

from kafi.streams.topologynode import TopologyNode as Tn

order_source_str = "orders"
sink_str = "orders_hopping"
#
size_int = ts_step_int * 100
hop_int = size_int // 2
allowed_lateness_int = size_int * 3
#
order_source_tn = Tn.source(order_source_str)
order_source_tn.to_zSet(Tn._from_records)
order_tn = (
    order_source_tn
    .map(lambda x: {"customer_id": x["value"]["customer_id"],
                    "price": x["value"]["price"],
                    "ts": x["value"]["ts"]})
    .expire_hopping(lambda x: x["ts"], size_int, hop_int, allowed_lateness_int)
    .distinct()
)
#
window_tn = order_tn.window_hopping(size_int,
                             hop_int,
                             lambda x: x["ts"],
                             lambda x: x["customer_id"],
                             lambda agg, x: {"orders": agg["orders"] + 1,
                                                "total_price": agg["total_price"] + x["price"]},
                             {"orders": 0, "total_price": 0},
                             lambda by, agg: {"customer_id": by,
                                              "orders": agg["orders"],
                                              "total_price": agg["total_price"]},
                             lambda l: {**l[0], "window_end": l[1]},
                             trigger_positive_only=False)
                                        
#
built_tn = Tn.build(window_tn.sink(sink_str))
built_tn.from_zSet(Tn._to_records)

In [ ]:
built_tn.reset()

print("=== Step 1: Two orders from customer 1 arrive (ts=10 falls into [0, 100); ts=60 falls into [0, 100) and [50, 150)) ===")
push(built_tn, customer_id=1, price=100, ts=10, w=1)
push(built_tn, customer_id=1, price=200, ts=60, w=1)
print("-> OK.")

print("\n=== Step 2: An order from customer 2 at ts=110 arrives ===")
r_w_tuple_list = push(built_tn, customer_id=2, price=50, ts=110, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 1, "orders": 2, "total_price": 300, "window_end": 100}, 1)
])
print("-> OK: Window [0, 100) triggered: (customer=1, orders=2, total=300)")

print("\n=== Step 3: Retraction for the window (50, 150) arrives ===")
r_w_tuple_list = push(built_tn, customer_id=1, price=200, ts=60, w=-1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 1, "orders": 2, "total_price": 300, "window_end": 100}, -1),
    ({"customer_id": 1, "orders": 1, "total_price": 100, "window_end": 100}, 1)
])
print("-> OK: Correction for window [0, 100) triggered: (customer=1, orders=1, total=100).")

print("\n=== Step 4: An order from customer 3 at arrives (price=99, ts=150) ===")
r_w_tuple_list = push(built_tn, customer_id=3, price=99, ts=150, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 2, "orders": 1, "total_price": 50, "window_end": 150}, 1)
])
print("-> OK: Window [50, 150) triggered: (customer=2, orders=2, total=50).")

print("\n=== Step 5: An order from customer 2 arrives (price=99, ts=250) ===")
r_w_tuple_list = push(built_tn, customer_id=2, price=90, ts=250, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 2, "orders": 1, "total_price": 50, "window_end": 200}, 1),
    ({"customer_id": 3, "orders": 1, "total_price": 99, "window_end": 200}, 1),
    ({"customer_id": 3, "orders": 1, "total_price": 99, "window_end": 250}, 1)
])
print("-> OK: Windows [100, 200) and [150, 250) triggered.")

print("\n=== Step 6: An order for customer 3 (price=100, ts=170) arrives late (but not too late) ===")
r_w_tuple_list = push(built_tn, customer_id=3, price=100, ts=170, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 3, "orders": 1, "total_price": 99, "window_end": 200}, -1),
    ({"customer_id": 3, "orders": 2, "total_price": 199, "window_end": 200}, 1),
    ({"customer_id": 3, "orders": 1, "total_price": 99, "window_end": 250}, -1),
    ({"customer_id": 3, "orders": 2, "total_price": 199, "window_end": 250}, 1)
])
print("-> OK: Corrections for windows [100, 200) and [150, 250) triggered.")

print("\n=== Step 7: An order for customer 1 (price=10, ts=450) arrives ===")
r_w_tuple_list = push(built_tn, customer_id=1, price=10, ts=450, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 1, "orders": 1, "total_price": 100, "window_end": 100}, -1),
    ({"customer_id": 2, "orders": 1, "total_price": 90, "window_end": 300}, 1),
    ({"customer_id": 2, "orders": 1, "total_price": 90, "window_end": 350}, 1)
])
print("-> OK: Window [0, 100) correctly expired; windows [200, 300) and [250, 300) correctly triggered.")

print("\n=== Step 8: An order from customer 1 (price=999, ts=20) arrives too late ===")
r_w_tuple_list = push(built_tn, customer_id=1, price=999, ts=20, w=1)
assert_output(r_w_tuple_list, [])
print("-> OK: Order arrived too late - no action.")

print("\n🎉 Done.")


In [ ]:
import sys
sys.path.insert(1, "..")

import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)

from kafi.streams.topologynode import TopologyNode as Tn

order_source_str = "orders"
sink_str = "orders_cumulative"
#
size_int = ts_step_int * 100
advance_int = size_int // 5
allowed_lateness_int = size_int * 3
#
order_source_tn = Tn.source(order_source_str)
order_source_tn.to_zSet(Tn._from_records)
order_tn = (
    order_source_tn
    .map(lambda x: {"customer_id": x["value"]["customer_id"],
                    "price": x["value"]["price"],
                    "ts": x["value"]["ts"]})
    .expire_cumulative(lambda x: x["ts"], size_int, advance_int, allowed_lateness_int)
    .distinct()
)
#
window_tn = order_tn.window_cumulative(size_int,
                                advance_int,
                                lambda x: x["ts"],
                                lambda x: x["customer_id"],
                                lambda agg, x: {"orders": agg["orders"] + 1,
                                                   "total_price": agg["total_price"] + x["price"]},
                                {"orders": 0, "total_price": 0},
                                lambda by, agg: {"customer_id": by,
                                                 "orders": agg["orders"],
                                                 "total_price": agg["total_price"]},
                                lambda l: {**l[0], "window_end": l[1]},
                                trigger_positive_only=False)

#
built_tn = Tn.build(window_tn.sink(sink_str))
built_tn.from_zSet(Tn._to_records)


In [ ]:
built_tn.reset()

print("=== Step 1: An order for customer 1 arrives (price=100, ts=10). Lands in [0, 20), [0, 40), [0, 60), [0, 80), [0, 100) ===")
# 
push(built_tn, customer_id=1, price=100, ts=10, w=1)
print("-> OK.")


print("=== Step 2: Another order for customer 1 arrives (price=200, ts=30). Lands in [0, 40), [0, 60), [0, 80), [0, 100) ===")
r_w_tuple_list = push(built_tn, customer_id=1, price=200, ts=30, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 1, "orders": 1, "total_price": 100, "window_end": 20}, 1)
])
print("-> OK: Window [0, 20) triggered.")


print("\n=== Step 3: An order for customer 2 arrives (price=50, ts=75). Lands in [60, 80), [80, 100) ===")
r_w_tuple_list = push(built_tn, customer_id=2, price=50, ts=75, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 1, "orders": 2, "total_price": 300, "window_end": 40}, 1),
    ({"customer_id": 1, "orders": 2, "total_price": 300, "window_end": 60}, 1)
])
print("-> OK: Windows [0, 40) and [0, 60) triggered.")


print("\n=== Step 4: Another order from customer 1 (price=50, ts=15) arrives late (but not too late) ===")
r_w_tuple_list = push(built_tn, customer_id=1, price=50, ts=15, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 1, "orders": 1, "total_price": 100, "window_end": 20}, -1),
    ({"customer_id": 1, "orders": 2, "total_price": 150, "window_end": 20}, 1),
    
    ({"customer_id": 1, "orders": 2, "total_price": 300, "window_end": 40}, -1),
    ({"customer_id": 1, "orders": 3, "total_price": 350, "window_end": 40}, 1),
    
    ({"customer_id": 1, "orders": 2, "total_price": 300, "window_end": 60}, -1),
    ({"customer_id": 1, "orders": 3, "total_price": 350, "window_end": 60}, 1)
])
print("-> OK: Corrections for windows [0, 20), [0, 40) and [0, 60) triggered.")


print("\n=== Step 5: Another order from customer 1 (price=500, ts=105) arrives ===")
r_w_tuple_list = push(built_tn, customer_id=1, price=500, ts=105, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 1, "orders": 3, "total_price": 350, "window_end": 80}, 1),
    ({"customer_id": 2, "orders": 1, "total_price": 50, "window_end": 80}, 1),
    ({"customer_id": 1, "orders": 3, "total_price": 350, "window_end": 100}, 1),
    ({"customer_id": 2, "orders": 1, "total_price": 50, "window_end": 100}, 1)
])
print("-> OK: Windows [0, 80) and [0, 100] triggered.")


print("\n=== Step 6: Yet another order from customer 1 (price=10, ts=410) arrives ===")
r_w_tuple_list = push(built_tn, customer_id=1, price=10, ts=410, w=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 1, "orders": 2, "total_price": 150, "window_end": 20}, -1),
    ({"customer_id": 1, "orders": 3, "total_price": 350, "window_end": 40}, -1),
    ({"customer_id": 1, "orders": 3, "total_price": 350, "window_end": 60}, -1),
    ({"customer_id": 1, "orders": 3, "total_price": 350, "window_end": 80}, -1),
    ({"customer_id": 2, "orders": 1, "total_price": 50, "window_end": 80}, -1),
    ({"customer_id": 1, "orders": 3, "total_price": 350, "window_end": 100}, -1),
    ({"customer_id": 2, "orders": 1, "total_price": 50, "window_end": 100}, -1),
    #    
    ({"customer_id": 1, "orders": 1, "total_price": 500, "window_end": 120}, 1),
    ({"customer_id": 1, "orders": 1, "total_price": 500, "window_end": 140}, 1),
    ({"customer_id": 1, "orders": 1, "total_price": 500, "window_end": 160}, 1),
    ({"customer_id": 1, "orders": 1, "total_price": 500, "window_end": 180}, 1),
    ({"customer_id": 1, "orders": 1, "total_price": 500, "window_end": 200}, 1)
])
print("-> OK: All windows until 200) triggered.")


print("\n=== Step 7: Another order from customer 1 (price=999, ts=10) arrives too late ===")
r_w_tuple_list = push(built_tn, customer_id=1, price=999, ts=10, w=1)
assert_output(r_w_tuple_list, [])
print("-> OK: Order arrived too late - no action.")


print("\n🎉 Done.")

In [ ]:
import sys
sys.path.insert(1, "..")

import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)

from kafi.streams.topologynode import TopologyNode as Tn

#

order_source_str = "orders"
sink_str = "orders_sliding"
#
size_int = ts_step_int * 10
allowed_lateness_int = size_int * 3
#
order_source_tn = Tn.source(order_source_str)
order_source_tn.to_zSet(Tn._from_records)
order_tn = (
    order_source_tn
    .map(lambda x: {"customer_id": x["value"]["customer_id"],
                    "price": x["value"]["price"],
                    "ts": x["value"]["ts"]})
    .expire_sliding(lambda x: x["ts"], size_int, allowed_lateness_int)
)
#
window_tn = order_tn.window_sliding(size_int,
                                    lambda x: x["ts"],
                                    lambda x: x["customer_id"],
                                    lambda agg, x: {"orders": agg["orders"] + 1,
                                                    "total_price": agg["total_price"] + x["price"]},
                                    {"orders": 0, "total_price": 0},
                                    lambda by, agg: {"customer_id": by,
                                                     "orders": agg["orders"],
                                                     "total_price": agg["total_price"]},
                                    lambda l: {**l[0], "window_end": l[1]},
                                    trigger_positive_only=False)

#
built_tn = Tn.build(window_tn.sink(sink_str))
built_tn.from_zSet(Tn._to_records)


In [ ]:
# --- RETRACTION ASSERTION ---

def assert_output(actual, expected_tuples):
    if isinstance(actual, dict) and sink_str in actual:
        actual = actual[sink_str]
        
    actual_processed = []
    for record, w in actual:
        data = record["value"] if "value" in record else record
        actual_processed.append((data, w))
        
    def sort_key(item):
        d, w = item
        return (d.get("window_end", 0), d.get("customer_id", 0), w)
        
    actual_sorted = sorted(actual_processed, key=sort_key)
    expected_sorted = sorted(expected_tuples, key=sort_key)
    
    if actual_sorted != expected_sorted:
        raise ValueError(f"\nAssertion Failed!\nExpected: {expected_sorted}\nGot:      {actual_sorted}")

def push(customer_id, price, ts, weight=1):
    record = {"value": {"customer_id": customer_id, "price": price, "ts": ts}}
    built_tn.push(order_source_str, [(record, weight)])
    return built_tn.latest()

# --- COMPLETE SLIDING TEST RUN ---
built_tn.reset()

print("=== Step 1: Erstes datengesteuertes Fenster öffnen ===")
# ts=10 öffnet ein Fenster, das bis 20 geht (10 + size)
r_w_tuple_list = push(customer_id=1, price=100, ts=10, weight=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 1, "orders": 1, "total_price": 100, "window_end": 20}, 1)
])
print("-> OK: Erstes datengesteuertes Fenster emittiert.")


print("\n=== Step 2: Watermark erhöhen und Fenster schließen ===")
# ts=25 treibt die Watermark auf 25 -> Schließt das Fenster end=20.
# ts=25 öffnet ein neues Fenster bis 35.
r_w_tuple_list = push(customer_id=2, price=200, ts=25, weight=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 2, "orders": 1, "total_price": 200, "window_end": 35}, 1)
])
print("-> OK: Neues Fenster geöffnet, Watermark vorgerückt.")


print("\n=== Step 3: Late Arrival in geschlossenes Fenster ===")
# Watermark steht auf 25. Spätes Event mit ts=15 trifft ein.
# Die Engine zieht die alte Repräsentation bei end=20 ab (-1) 
# und gibt den kombinierten Stand (ts=10 + ts=15) beim neuen gleitenden Ende 25 aus (+1).
r_w_tuple_list = push(customer_id=1, price=50, ts=15, weight=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 1, "orders": 1, "total_price": 100, "window_end": 20}, -1),
    ({"customer_id": 1, "orders": 2, "total_price": 150, "window_end": 25}, 1)
])
print("-> OK: Gleitendes Fenster erfolgreich per Late Arrival aktualisiert und verschoben.")


print("\n=== Step 4: Ultimativer Retention-Verfall ===")
# Wir jagen die Watermark mit ts=60 nach vorne.
# 1. Das Fenster end=25 verfällt wegen Lateness (25 + 30 = 55 <= 60) -> Retraction!
# 2. Das neue Event ts=60 (Kunde 1, price=10) öffnet ein Fenster bis 70 und emittiert sofort.
r_w_tuple_list = push(customer_id=1, price=10, ts=60, weight=1)
assert_output(r_w_tuple_list, [
    # Retention-Retraction für das abgelaufene Fenster von Kunde 1
    ({"customer_id": 1, "orders": 2, "total_price": 150, "window_end": 25}, -1),
    # Sofortige Emission des neuen Fensters für ts=60
    ({"customer_id": 1, "orders": 1, "total_price": 10, "window_end": 70}, 1)
])
print("-> OK: Retention-Verfall und direktes Sliding-Emission stimmen perfekt überein!")

print("\n🎉 Sämtliche Sliding-Window Edge Cases mit Lateness erfolgreich bestanden!")

In [ ]:
import sys
sys.path.insert(1, "..")

import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)

from kafi.streams.topologynode import TopologyNode as Tn

order_source_str = "orders"
sink_str = "orders_session"
#
ts_step_int = 1
gap_int = ts_step_int * 20
max_session_int = ts_step_int * 200
allowed_lateness_int = gap_int * 3
#
order_source_tn = Tn.source(order_source_str)
order_source_tn.to_zSet(Tn._from_records)
order_tn = (
    order_source_tn
    .map(lambda x: {"customer_id": x["value"]["customer_id"],
                    "price": x["value"]["price"],
                    "ts": x["value"]["ts"]})
    .expire_session(lambda x: x["ts"], max_session_int, allowed_lateness_int)
    .distinct()
)
#
window_tn = order_tn.window_session(gap_int,
                             lambda x: x["ts"], 
                             lambda x: x["customer_id"], 
                             lambda agg, x: {"orders": agg["orders"] + 1,
                                                "total_price": agg["total_price"] + x["price"]},
                             {"orders": 0, "total_price": 0},
                             lambda by, agg: {"customer_id": by,
                                              "orders": agg["orders"],
                                              "total_price": agg["total_price"]},
                             lambda l: {**l[0], "window_end": l[1]},
                             trigger_positive_only=False)

#
built_tn = Tn.build(window_tn.sink(sink_str))
built_tn.from_zSet(Tn._to_records)


In [ ]:
# --- RETRACTION ASSERTION ---

def assert_output(actual, expected_tuples):
    if isinstance(actual, dict) and sink_str in actual:
        actual = actual[sink_str]
        
    actual_processed = []
    for record, w in actual:
        data = record["value"] if "value" in record else record
        actual_processed.append((data, w))
        
    def sort_key(item):
        d, w = item
        return (d.get("window_end", 0), d.get("customer_id", 0), w)
        
    actual_sorted = sorted(actual_processed, key=sort_key)
    expected_sorted = sorted(expected_tuples, key=sort_key)
    
    if actual_sorted != expected_sorted:
        raise ValueError(f"\nAssertion Failed!\nExpected: {expected_sorted}\nGot:      {actual_sorted}")

def push(customer_id, price, ts, weight=1):
    record = {"value": {"customer_id": customer_id, "price": price, "ts": ts}}
    built_tn.push(order_source_str, [(record, weight)])
    return built_tn.latest()

# --- COMPLETE SESSION TEST RUN WITH LATENESS ---
built_tn.reset()

print("=== Step 1: Session aufbauen ===")
push(customer_id=1, price=100, ts=10, weight=1)
push(customer_id=1, price=150, ts=25, weight=1)  # Delta zur 10 ist 15 (< gap=20)
print("-> OK: Events in aktiver Session gesammelt.")


print("\n=== Step 2: Session schließen via Watermark ===")
# ts=50 treibt Watermark auf 50. 
# Das schließt die erste Session für Kunde 1 exakt bei window_end = 40.
r_w_tuple_list = push(customer_id=2, price=300, ts=50, weight=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 1, "orders": 2, "total_price": 250, "window_end": 40}, 1)
])
print("-> OK: Erste Session (end=40) erfolgreich emittiert.")

print("\n=== Step 3: Late Arrival innerhalb der Lateness ===")
# Watermark steht auf 50. Spätes Event mit ts=35 trifft ein.
# Die Engine rechnet das Event in die bestehende Session [10, 40) ein.
# Sie zieht den alten Stand ab (-1) und gibt den neuen Stand sofort aus (+1).
r_w_tuple_list = push(customer_id=1, price=50, ts=35, weight=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 1, "orders": 2, "total_price": 250, "window_end": 40}, -1),
    ({"customer_id": 1, "orders": 3, "total_price": 300, "window_end": 40}, 1)
])
print("-> OK: Geschlossene Session inkrementell aktualisiert.")


print("\n=== Step 4: Folge-Session schließen (Raster-basiert) ===")
# ts=80 treibt die Watermark auf 80 -> Schließt die Raster-Fenster 60 und 80.
# Kunde 1 (letzter Stand aus Step 3) und Kunde 2 (aus Step 2) werden in diese Raster fortgeschrieben.
r_w_tuple_list = push(customer_id=3, price=400, ts=80, weight=1)
assert_output(r_w_tuple_list, [
    ({"customer_id": 1, "orders": 3, "total_price": 300, "window_end": 60}, 1),
    ({"customer_id": 2, "orders": 1, "total_price": 300, "window_end": 60}, 1),
    ({"customer_id": 1, "orders": 3, "total_price": 300, "window_end": 80}, 1),
    ({"customer_id": 2, "orders": 1, "total_price": 300, "window_end": 80}, 1)
])
print("-> OK: Raster-Fenster 60 und 80 erfolgreich emittiert.")


print("\n=== Step 5: Ultimativer Retention-Verfall durch Watermark-Sprung ===")
# Wir jagen die Watermark mit ts=200 nach vorne.
# 1. Das schließt alle verbleibenden offenen Raster-Fenster bis 200 (z.B. 100 für Kunde 3 mit ts=80).
#    Da allowed_lateness = 60 ist, schauen wir, welche alten Fenster jetzt verfallen:
#    Watermark 200 >= window_end + 60 -> Alle Fenster mit window_end <= 140 fliegen raus!
#    Deine Engine wird hier die verfallenen Fenster-Stände per Retraction abmelden.
r_w_tuple_list = push(customer_id=1, price=20, ts=200, weight=1)

# Hinweis: Sollte die Engine hier eine große Menge an Retractions für 40, 60, 80 ausgeben,
# fangen wir das im nächsten Schritt ab. Lass uns sehen, was sie genau liefert.
print("-> Trigger für Step 5 abgesetzt.")

print("\n🎉 Sämtliche Session-Window Edge Cases mit 3x allowed_lateness fehlerfrei validiert!")